# **Text Classification with RNN: Sentiment Analysis on IMDB Movie Reviews**


In [ ]:
# Google Colab setup — ensures local datasets resolve correctly
import os
from pathlib import Path

def setup_notebook_dir(marker_files=None):
    """Change working directory to the folder containing marker files."""
    marker_files = marker_files or []
    search_roots = [
        Path.cwd(),
        Path('/content'),
        Path('/content/Deep-Learning-Workshop-2026'),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for path in [root] + [p for p in root.rglob('*') if p.is_dir()]:
            if marker_files and all((path / m).exists() for m in marker_files):
                os.chdir(path)
                print(f'Working directory: {path}')
                return path
    print(f'Working directory: {Path.cwd()}')
    return Path.cwd()

setup_notebook_dir(['IMDB Dataset.csv'])

In [1]:
# Import essential libraries for data processing, visualization, and deep learning
import numpy as np               # Array operations and numerical computations
import pandas as pd              # Data manipulation and analysis
import matplotlib.pyplot as plt  # Data visualization
import seaborn as sns            # Advanced data visualization
import torch                     # PyTorch deep learning framework
import torch.nn as nn            # Neural network modules
from torch.utils.data import DataLoader, TensorDataset  # Data loading utilities
from collections import Counter  # Frequency counting
from string import punctuation   # Punctuation character set
import re                        # Python Regex

# Import additional libraries
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

# Set random seeds for reproducible results
torch.manual_seed(42)
np.random.seed(42)

Using cuda device


## Q1: Load the IMDB dataset (`IMDB Dataset.csv`) using pandas and display the first 20 rows

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df.head(20)

## Q2: Perform basic preprocessing on the review text: convert all text to uppercase, remove all punctuations.

In [ ]:
# Convert to uppercase and remove punctuation
df["clean_text"] = (
    df["review"]
    .str.upper()
    .str.replace(r"<.*?>", "", regex=True)
    .apply(lambda x: "".join(" " if c in punctuation else c for c in x))
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df[["review", "clean_text"]].head()

## Q3: Convert the sentiment labels (“positive”, “negative”) into numerical form suitable for binary classification.

In [ ]:
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
df[["sentiment", "label"]].head()

## Q4: Build a vocabulary from the cleaned reviews and create a word-to-index dictionary. Print the size of the vocabulary and the integer mapping for any five sample words.

In [ ]:
all_text = " ".join(df["clean_text"].tolist())
words = all_text.split()
word_counts = Counter(words)

sorted_words = word_counts.most_common()
vocab_to_int = {word: idx + 1 for idx, (word, _) in enumerate(sorted_words)}

print(f"Vocabulary size: {len(vocab_to_int):,}")
print("\nSample word-to-index mappings:")
for word, idx in list(vocab_to_int.items())[:5]:
    print(f"  {word}: {idx}")

## Q5: Convert each review into a sequence of integers, such that: 
- Use a fixed sequence length of 250:
- truncate longer reviews from the end,
- pad shorter reviews with zeros at the start.

In [ ]:
def pad_features(reviews_int, seq_length):
    features = np.zeros((len(reviews_int), seq_length), dtype=int)
    for i, review in enumerate(reviews_int):
        if len(review) <= seq_length:
            padding = [0] * (seq_length - len(review))
            features[i, :] = np.array(padding + review)
        else:
            features[i, :] = np.array(review[-seq_length:])
    return features

seq_length = 250
reviews_encoded = [
    [vocab_to_int[word] for word in review.split() if word in vocab_to_int]
    for review in df["clean_text"]
]
encoded_labels = df["label"].to_numpy(dtype=np.float32)

padded_features = pad_features(reviews_encoded, seq_length)
padded_features.shape

## Q6: Split the dataset into training, validation, and test sets in the ratio 70:15:15. Then create DataLoaders with a batch size of 64.

In [ ]:
total_samples = len(padded_features)
train_end = int(0.70 * total_samples)
val_end = train_end + int(0.15 * total_samples)

X_train = padded_features[:train_end]
X_val = padded_features[train_end:val_end]
X_test = padded_features[val_end:]

y_train = encoded_labels[:train_end]
y_val = encoded_labels[train_end:val_end]
y_test = encoded_labels[val_end:]

train_data = TensorDataset(torch.from_numpy(X_train).long(), torch.from_numpy(y_train))
val_data = TensorDataset(torch.from_numpy(X_val).long(), torch.from_numpy(y_val))
test_data = TensorDataset(torch.from_numpy(X_test).long(), torch.from_numpy(y_test))

batch_size = 64
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## Q7: Define a simple RNN-based sentiment classifier using:

- embedding dimension = 128,
- hidden dimension = 64,
- 2 recurrent layers,
- dropout = 0.3.

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            n_layers,
            batch_first=True,
            dropout=drop_prob if n_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, hidden = self.rnn(embedded)
        hidden = self.dropout(hidden[-1])
        return self.sigmoid(self.fc(hidden))


vocab_size = len(vocab_to_int) + 1
output_size = 1
embedding_dim = 128
hidden_dim = 64
n_layers = 2
drop_prob = 0.3

rnn_model = SentimentRNN(
    vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob
).to(device)

## Q8: Train the RNN model for 5 epochs using:

- binary cross-entropy loss,
- Adam optimizer,
- learning rate = 0.0005.

Plot the training and validation loss curves.

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score


def calculate_metrics(predictions, labels):
    binary_preds = (predictions >= 0.5).astype(int)
    accuracy = accuracy_score(labels, binary_preds)
    f1 = f1_score(labels, binary_preds)
    return {"accuracy": accuracy, "f1": f1}


def evaluate(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.inference_mode():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()
            outputs = model(inputs)
            loss = criterion(outputs.squeeze(), labels)
            total_loss += loss.item()
            all_preds.extend(outputs.squeeze().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    metrics = calculate_metrics(np.array(all_preds), np.array(all_labels))
    return avg_loss, metrics


criterion = nn.BCELoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=0.0005)
epochs = 5

train_losses, val_losses = [], []

for epoch in range(epochs):
    rnn_model.train()
    running_loss = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = rnn_model(inputs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    val_loss, _ = evaluate(rnn_model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, marker="o", label="Train Loss")
plt.plot(range(1, epochs + 1), val_losses, marker="s", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RNN Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## Q9: Replace the RNN layer with an LSTM layer while keeping the rest of the architecture similar. Train the LSTM model and compare:
- test accuracy,
- F1-score,
with the RNN model. Explain which architecture is more suitable for sentiment analysis and why.

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            n_layers,
            batch_first=True,
            dropout=drop_prob if n_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = self.dropout(hidden[-1])
        return self.sigmoid(self.fc(hidden))


lstm_model = SentimentLSTM(
    vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob
).to(device)

optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.0005)

for epoch in range(epochs):
    lstm_model.train()
    running_loss = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float()

        optimizer_lstm.zero_grad()
        outputs = lstm_model(inputs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer_lstm.step()
        running_loss += loss.item()

    val_loss, _ = evaluate(lstm_model, val_loader, criterion)
    print(f"LSTM Epoch {epoch + 1}/{epochs} | Train Loss: {running_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

_, rnn_test_metrics = evaluate(rnn_model, test_loader, criterion)
_, lstm_test_metrics = evaluate(lstm_model, test_loader, criterion)

print("\nTest Set Comparison:")
print(f"RNN  -> Accuracy: {rnn_test_metrics['accuracy']:.4f}, F1: {rnn_test_metrics['f1']:.4f}")
print(f"LSTM -> Accuracy: {lstm_test_metrics['accuracy']:.4f}, F1: {lstm_test_metrics['f1']:.4f}")

if lstm_test_metrics["f1"] > rnn_test_metrics["f1"]:
    print("\nLSTM performs better because it captures long-range dependencies in reviews using gated memory cells.")
else:
    print("\nRNN performs comparably on this task, but LSTM is generally more stable for longer sequences.")